In [2]:
import requests
import pandas as pd
import sqlite3

# --- CONFIGURATION ---
# PLEASE REPLACE THE VALUE BELOW WITH YOUR OWN TMDB API KEY
TMDB_API_KEY = "YOUR_API_KEY_HERE"

# Task 1: Build the Pipeline
def build_pipeline():
    url = f"https://api.themoviedb.org/3/discover/movie?api_key={TMDB_API_KEY}&language=en-US&sort_by=popularity.desc&include_adult=false&include_video=false&page=1"

    response = requests.get(url)
    if response.status_code != 200:
        print(f"Error fetching data: {response.status_code}")
        return

    data = response.json()['results']
    df = pd.DataFrame(data)

    # Store in SQLite database
    conn = sqlite3.connect('movies_data.db')
    df.to_sql('movies', conn, if_exists='replace', index=False)
    conn.close()
    print("Pipeline: Data successfully pulled and saved to SQLite table 'movies'.")

# Task 2: Perform EDA
def perform_eda():
    conn = sqlite3.connect('movies_data.db')
    df_loaded = pd.read_sql_query("SELECT * FROM movies", conn)
    conn.close()

    print("\n--- EDA: First 5 Rows ---")
    print(df_loaded.head())

    print("\n--- EDA: Summary Statistics ---")
    print(df_loaded.describe())

    print("\n--- EDA: Movie Count per Genre ID (Top level) ---")
    # Note: genre_ids is stored as a list/string in the API response
    # We'll explode it to count individual genre occurrences if it's in list format
    # or simply count the primary genre ID provided.
    genre_counts = df_loaded['genre_ids'].apply(lambda x: eval(x)[0] if isinstance(x, str) and eval(x) else None).value_counts()
    print(genre_counts)

    print("\n--- EDA: Missing Values ---")
    print(df_loaded.isnull().sum())

if __name__ == "__main__":
    build_pipeline()
    perform_eda()

Error fetching data: 401


DatabaseError: Execution failed on sql 'SELECT * FROM movies': no such table: movies